In [2]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import faiss

In [3]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense
from tensorflow.keras.applications.resnet50 import preprocess_input

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [35]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
TOP_K = 5

MODEL_PATH = "../models/brand_classifier_resnet50.keras"

TRAIN_CSV = "../datasets/splits/train_images.csv"
VAL_CSV = "../datasets/splits/val_images.csv"
TEST_CSV = "../datasets/splits/test_images.csv"

OUTPUT_EMBEDDINGS = "../embeddings/shoe_image_embeddings_finetuned_v1.npy"
OUTPUT_FILENAMES = "../embeddings/shoe_image_filenames_v1.npy"
OUTPUT_INDEX = "../embeddings/shoe_faiss_index_v1.index"

In [6]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 1848
Val: 397
Test: 397


In [7]:
print(train_df.head())
print(train_df.columns.tolist())

                                          image_path  \
0  ../datasets/sampled_sneakers_1024x1024\resized...   
1  ../datasets/sampled_sneakers_1024x1024\normali...   
2  ../datasets/sampled_sneakers_1024x1024\resized...   
3  ../datasets/sampled_sneakers_1024x1024\resized...   
4  ../datasets/sampled_sneakers_1024x1024\resized...   

                           image_name        brand  
0         resized_padded_vans_167.jpg         vans  
1              normalized_nike_15.jpg         nike  
2       resized_padded_reebok_415.jpg       reebok  
3            resized_all_vans_575.jpg         vans  
4  resized_padded_underarmour_269.jpg  underarmour  
['image_path', 'image_name', 'brand']


In [8]:
def resolve_path(p):
    if os.path.exists(p):
        return p

    if os.path.exists(os.path.join("..", p)):
        return os.path.join("..", p)

    return p


for df in [train_df, val_df, test_df]:
    df["image_path"] = df["image_path"].apply(resolve_path)

print("Missing train:",
      (~train_df["image_path"].apply(os.path.exists)).sum())

print("Missing val:",
      (~val_df["image_path"].apply(os.path.exists)).sum())

print("Missing test:",
      (~test_df["image_path"].apply(os.path.exists)).sum())

Missing train: 0
Missing val: 0
Missing test: 0


In [9]:
classifier = tf.keras.models.load_model(MODEL_PATH)

classifier.summary()

Model: "ResNet50_BrandClassifier"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 resnet50 (Functional)       (None, 7, 7, 2048)        23587712  
                                                                 
 global_average_pooling2d (G  (None, 2048)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 256)               524544    
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 6)                 1542      
                                          

In [10]:
dense_layers = [
    layer for layer in classifier.layers
    if isinstance(layer, Dense)
]

for layer in dense_layers:
    print(
        layer.name,
        "units =", layer.units,
        "activation =", layer.activation.__name__
    )

dense units = 256 activation = relu
dense_1 units = 6 activation = softmax


In [12]:
embedding_layer = None

for layer in dense_layers:
    if layer.units == 256:
        embedding_layer = layer
        break

print("Using embedding layer:", embedding_layer.name)

embedding_model = Model(
    inputs=classifier.input,
    outputs=embedding_layer.output
)

embedding_model.summary()

Using embedding layer: dense
Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 resnet50 (Functional)       (None, 7, 7, 2048)        23587712  
                                                                 
 global_average_pooling2d (G  (None, 2048)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 256)               524544    
                                                                 
Total params: 24,112,256
Trainable params: 14,974,720
Non-trainable params: 9,137,536
_________________________________________________________________


In [13]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.cast(img, tf.float32)
    img = preprocess_input(img)
    return img

In [14]:
def make_dataset(df):
    paths = df["image_path"].values

    ds = tf.data.Dataset.from_tensor_slices(paths)

    ds = ds.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

In [15]:
train_ds = make_dataset(train_df)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

In [16]:
def extract_embeddings(model, dataset):
    emb = model.predict(
        dataset,
        verbose=1
    )

    return emb.astype("float32")

In [17]:
train_embeddings = extract_embeddings(
    embedding_model,
    train_ds
)

print("Train embedding shape:", train_embeddings.shape)

116/116 [==============================] - 11s 52ms/step
Train embedding shape: (1848, 256)


In [18]:
val_embeddings = extract_embeddings(
    embedding_model,
    val_ds
)

print("Val embedding shape:", val_embeddings.shape)

25/25 [==============================] - 1s 58ms/step
Val embedding shape: (397, 256)


In [19]:
test_embeddings = extract_embeddings(
    embedding_model,
    test_ds
)

print("Test embedding shape:", test_embeddings.shape)

25/25 [==============================] - 1s 52ms/step
Test embedding shape: (397, 256)


In [20]:
def normalize_embeddings(x):
    norms = np.linalg.norm(
        x,
        axis=1,
        keepdims=True
    )

    return x / np.maximum(norms, 1e-12)

In [21]:
train_embeddings = normalize_embeddings(train_embeddings)
val_embeddings = normalize_embeddings(val_embeddings)
test_embeddings = normalize_embeddings(test_embeddings)

In [22]:
print(
    "First embedding norm:",
    np.linalg.norm(train_embeddings[0])
)

First embedding norm: 1.0


In [23]:
embedding_dim = train_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)

index.add(
    train_embeddings.astype("float32")
)

print("Embedding dimension:", embedding_dim)
print("FAISS vectors:", index.ntotal)

Embedding dimension: 256
FAISS vectors: 1848


In [36]:
os.makedirs(
    os.path.dirname(OUTPUT_INDEX),
    exist_ok=True
)

faiss.write_index(
    index,
    OUTPUT_INDEX
)

np.save(
    OUTPUT_EMBEDDINGS,
    train_embeddings
)

np.save(
    OUTPUT_FILENAMES,
    train_df["image_name"].values
)

print("Saved:")
print(OUTPUT_INDEX)
print(OUTPUT_EMBEDDINGS)
print(OUTPUT_FILENAMES)

Saved:
../embeddings/shoe_faiss_index_v1.index
../embeddings/shoe_image_embeddings_finetuned_v1.npy
../embeddings/shoe_image_filenames_v1.npy


In [25]:
def retrieve(query_embeddings, k=5):
    scores, indices = index.search(
        query_embeddings.astype("float32"),
        k
    )

    return scores, indices

In [26]:
scores, indices = retrieve(
    test_embeddings[:1],
    TOP_K
)

print("Scores:")
print(scores[0])

print("\nRetrieved images:")

for idx in indices[0]:
    print(train_df.iloc[idx]["image_name"],
          "|",
          train_df.iloc[idx]["brand"])

Scores:
[0.89623433 0.8760411  0.87306964 0.8673635  0.8642876 ]

Retrieved images:
resized_padded_new_newbalance_158.jpg | newbalance
resized_padded_new_newbalance_258.jpg | newbalance
resized_padded_new_newbalance_20.jpg | newbalance
resized_padded_new_newbalance_45.jpg | newbalance
resized_padded_new_newbalance_13.jpg | newbalance


In [27]:
def evaluate_retrieval(
    query_embeddings,
    query_df,
    k=5
):
    scores, indices = retrieve(
        query_embeddings,
        k
    )

    y_true = query_df["brand"].values

    top1_correct = 0
    top5_correct = 0
    majority_correct = 0

    for i in range(len(query_df)):

        true_brand = y_true[i]

        retrieved_brands = [
            train_df.iloc[idx]["brand"]
            for idx in indices[i]
        ]

        # Top-1
        if retrieved_brands[0] == true_brand:
            top1_correct += 1

        # Top-5
        if true_brand in retrieved_brands:
            top5_correct += 1

        # Majority
        counts = pd.Series(
            retrieved_brands
        ).value_counts()

        majority_brand = counts.index[0]

        if majority_brand == true_brand:
            majority_correct += 1

    n = len(query_df)

    return {
        "top1": top1_correct / n,
        "top5": top5_correct / n,
        "majority": majority_correct / n
    }

In [28]:
val_results = evaluate_retrieval(
    val_embeddings,
    val_df,
    TOP_K
)

print("Brand Classifier Embedding — Validation")
print("-----------------------------------------")
print(f"Top-1:    {val_results['top1']:.4%}")
print(f"Top-5:    {val_results['top5']:.4%}")
print(f"Majority: {val_results['majority']:.4%}")

Brand Classifier Embedding — Validation
-----------------------------------------
Top-1:    90.1763%
Top-5:    93.9547%
Majority: 89.6725%


In [29]:
test_results = evaluate_retrieval(
    test_embeddings,
    test_df,
    TOP_K
)

print("Brand Classifier Embedding — Test")
print("----------------------------------")
print(f"Top-1:    {test_results['top1']:.4%}")
print(f"Top-5:    {test_results['top5']:.4%}")
print(f"Majority: {test_results['majority']:.4%}")

Brand Classifier Embedding — Test
----------------------------------
Top-1:    90.6801%
Top-5:    93.9547%
Majority: 90.1763%


In [30]:
def per_brand_retrieval(
    query_embeddings,
    query_df,
    k=5
):
    scores, indices = retrieve(
        query_embeddings,
        k
    )

    results = []

    for brand in sorted(query_df["brand"].unique()):

        mask = (
            query_df["brand"].values == brand
        )

        total = mask.sum()

        top1 = 0
        top5 = 0
        majority = 0

        for i in np.where(mask)[0]:

            retrieved_brands = [
                train_df.iloc[idx]["brand"]
                for idx in indices[i]
            ]

            if retrieved_brands[0] == brand:
                top1 += 1

            if brand in retrieved_brands:
                top5 += 1

            majority_brand = (
                pd.Series(retrieved_brands)
                .value_counts()
                .index[0]
            )

            if majority_brand == brand:
                majority += 1

        results.append({
            "brand": brand,
            "top1": top1 / total,
            "top5": top5 / total,
            "majority": majority / total,
            "support": total
        })

    return pd.DataFrame(results)

In [31]:
brand_results = per_brand_retrieval(
    test_embeddings,
    test_df,
    TOP_K
)

brand_results

,brand,top1,top5,majority,support
0,newbalance,0.941176,0.941176,0.941176,34
1,nike,0.964912,0.982456,0.982456,57
2,puma,0.829268,0.890244,0.780488,82
3,reebok,0.764706,0.882353,0.803922,51
4,underarmour,1.000000,1.000000,1.000000,35
5,vans,0.949275,0.956522,0.942029,138


In [34]:
EMBEDDING_MODEL_PATH = "../models/embedding_model_finetuned.keras"

embedding_model.save(EMBEDDING_MODEL_PATH)